# Conversational LLM — Decoder-only architecture

This notebook enables the user to experiment with the inference of a trained LLM on the 85M token French chats dataset (https://github.com/GaetanKlopocki/French_LLM_decoder , forked from https://github.com/angeluriot/French_instruct).
The model was trained on 60,000 epochs (~ 5 days of training on a single Colab GPU).

**Note for GitHub Users:** This notebook is specifically designed for **inferencing with a trained model**. A separate notebook dedicated to training your own LLM is available on the GitHub repo.


**Notebook Structure:**
1. GPU Check
2. Configuration
3. Google Drive (persistent checkpoints)
4. Model and tokenizer loading
4. Model Initialization
5. Generation / Interactive Chat (with KV cache)

**⚠️ Realistic Expectations:** An LLM built "from scratch" and trained on a single Colab GPU (T4 free, ~16 GB) remains a pedagogical project—a few tens of millions of parameters, far from commercial LLMs.

For this model, the architecture and the training time enable the model to create sentences with a readable syntax and words within the context of the prompt, but it falls short of properly answering the prompt.

## 1. GPU Check

In [1]:
import torch

print("PyTorch :", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

PyTorch : 2.11.0+cpu
Device utilisé : cpu


## 2. Configuration

In [2]:
CONFIG = {
    # --- Données ---
    "max_history_turns": 6,
    "prompt_max_len": 192,
    "response_max_len": 128,

    # --- Tokenizer ---
    "vocab_size": 16000,

    # --- Model ---
    "d_model": 384,
    "n_heads": 6,
    "n_layers": 6,
    "d_ff": 1024,
    "dropout": 0.1,
    "rope_theta": 10000.0,
    "pos_max_len": 1024,
}

## 3. Model and tokenizer loading

In [8]:
import os
from huggingface_hub import hf_hub_download

REPO_DIR = "/content/French_LLM_Decoder"
MODEL_FILENAME = "French_LLM_decoder_trained.pt"
INFERENCE_SUBDIR = "Inference"

# Clone GitHub repo
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/GaetanKlopocki/French_LLM_Decoder.git {REPO_DIR}

inference_path = os.path.join(REPO_DIR, INFERENCE_SUBDIR)
os.makedirs(inference_path, exist_ok=True)

# Read HF_REPO_ID from the specified file
hf_repo_id_file_path = os.path.join(inference_path, "model_path")
if os.path.exists(hf_repo_id_file_path):
    with open(hf_repo_id_file_path, "r") as f:
        HF_REPO_ID = f.read().strip()
    print(f"HF_REPO_ID loaded from Hugging Face: {HF_REPO_ID}")
else:
    # Fallback to default if file not found
    HF_REPO_ID = "GaetanKlopocki/French_LLM_decoder"
    print(f"File {hf_repo_id_file_path} not found. Using default HF_REPO_ID: {HF_REPO_ID}")

# Download trained model from Hugging Face
model_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename=MODEL_FILENAME,
    local_dir=inference_path,
)
print(f"Model ready at {model_path}")

HF_REPO_ID loaded from Hugging Face: GaetanKlopocki/French_LLM_decoder
Model ready at /content/French_LLM_Decoder/Inference/French_LLM_decoder_trained.pt


In [4]:
import os
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

tokenizer_path = "/content/French_LLM_Decoder/Inference/tokenizer_trained.json"

if os.path.exists(tokenizer_path):
    print("Tokenizer found")
    tokenizer = Tokenizer.from_file(tokenizer_path)
    SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>", "<user>", "<assistant>", "<ctx>"]
    PAD, BOS, EOS, UNK, USR, AST, CTX = SPECIAL_TOKENS
    PAD_ID = tokenizer.token_to_id(PAD)
    BOS_ID = tokenizer.token_to_id(BOS)
    EOS_ID = tokenizer.token_to_id(EOS)
    VOCAB_SIZE = tokenizer.get_vocab_size()
else:
    print("Tokenizer not found")

Tokenizer found


## 4. Model initiation

In [5]:
import math
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        norm = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(norm + self.eps)
        return x * self.weight


def precompute_rope_cos_sin(dim, max_len, theta=10000.0):
    """Precompute cos/sin for RoPE."""
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))  # (dim/2,)
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)          # (max_len, dim/2)
    emb = torch.cat([freqs, freqs], dim=-1)   # (max_len, dim)
    return emb.cos(), emb.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    # x: (B, H, L, d_k) ; cos/sin: (L, d_k)
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)


def make_causal_mask(q_len, k_len, past_len, device):
    # mask[i, j] = True if the query i (absolute position = past_len+i) can see the key j
    i = torch.arange(q_len, device=device).unsqueeze(1)
    j = torch.arange(k_len, device=device).unsqueeze(0)
    allowed = j <= (past_len + i)
    return allowed.unsqueeze(0).unsqueeze(0)  # (1,1,q_len,k_len)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask, cos, sin, kv_cache=None):
        B, L, _ = x.shape
        q = self.w_q(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        k = self.w_k(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        v = self.w_v(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if kv_cache is not None and kv_cache[0] is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v) if kv_cache is not None else None

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores.masked_fill(~mask, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, L, -1)
        return self.w_o(out), new_cache


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)  # gate
        self.w2 = nn.Linear(d_model, d_ff, bias=False)  # value
        self.w3 = nn.Linear(d_ff, d_model, bias=False)  # exit
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))


class Block(nn.Module):
    """Causal Self-attention (Pre-RMSNorm) + SwiGLU (Pre-RMSNorm)."""

    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, mask, cos, sin, kv_cache=None):
        h, new_cache = self.attn(self.norm1(x), mask, cos, sin, kv_cache)
        x = x + h
        x = x + self.ff(self.norm2(x))
        return x, new_cache


class DecoderTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=384, n_heads=6, n_layers=6, d_ff=1024,
                 max_len=1024, dropout=0.1, pad_id=0, rope_theta=10000.0):
        super().__init__()
        self.pad_id = pad_id
        self.d_k = d_model // n_heads
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.embed.weight

        cos, sin = precompute_rope_cos_sin(self.d_k, max_len, rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, ids, kv_cache=None):
        B, L = ids.shape
        device = ids.device
        if kv_cache is not None and kv_cache[0][0] is not None:
            past_len = kv_cache[0][0].shape[2]
        else:
            past_len = 0

        x = self.drop(self.embed(ids))
        cos = self.rope_cos[past_len:past_len + L].to(device)
        sin = self.rope_sin[past_len:past_len + L].to(device)
        mask = make_causal_mask(L, past_len + L, past_len, device)

        new_caches = []
        for i, block in enumerate(self.blocks):
            layer_cache = kv_cache[i] if kv_cache is not None else None
            x, new_cache = block(x, mask, cos, sin, layer_cache)
            new_caches.append(new_cache)

        x = self.norm_f(x)
        logits = self.head(x)
        return logits, new_caches

    def init_cache(self):
        return [(None, None) for _ in self.blocks]

    @torch.no_grad()
    def generate(self, prompt_ids, bos_id, eos_id, max_new_tokens=100, temperature=0.8, top_k=50):
        """Generate one sequence (no batching) from a tokenized prompt (ids list, no <bos>)."""
        self.eval()
        device = next(self.parameters()).device
        ids = torch.tensor([[bos_id] + list(prompt_ids)], dtype=torch.long, device=device)

        logits, cache = self.forward(ids, kv_cache=self.init_cache())
        next_logits = logits[:, -1, :] / temperature
        generated = []

        for _ in range(max_new_tokens):
            step_logits = next_logits.clone()
            if top_k is not None:
                v, _ = torch.topk(step_logits, top_k)
                step_logits = step_logits.masked_fill(step_logits < v[:, [-1]], float("-inf"))
            probs = torch.softmax(step_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # (1,1)
            token_id = next_token.item()
            if token_id == eos_id:
                break
            generated.append(token_id)

            logits, cache = self.forward(next_token, kv_cache=cache)
            next_logits = logits[:, -1, :] / temperature

        self.train()
        return generated

In [6]:
model = DecoderTransformer(
    vocab_size=CONFIG["vocab_size"],
    d_model=CONFIG["d_model"],
    n_heads=CONFIG["n_heads"],
    n_layers=CONFIG["n_layers"],
    d_ff=CONFIG["d_ff"],
    max_len=CONFIG["pos_max_len"],
    dropout=CONFIG["dropout"],
    pad_id=PAD_ID,
    rope_theta=CONFIG["rope_theta"],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {n_params:,}".replace(",", " "))

Number of parameters: 16 765 824


In [7]:
import math, time
import torch.nn as nn
import os

model_path = "/content/French_LLM_Decoder/Inference/French_LLM_decoder_trained.pt"

if os.path.exists(model_path):
    print(f"Model found at {model_path}")
    ckpt = torch.load(model_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    print("Model loaded successfully.")
else:
    print(f"Model not found at {model_path}.")

Model found at /content/French_LLM_Decoder/Inference/French_LLM_decoder_trained.pt
Model loaded successfully.


## 5. Text generation / interactive chat (with KV cache)

In [ ]:
def format_history(history, max_history_turns=CONFIG["max_history_turns"], ctx=""):
    hist = history[-max_history_turns:]
    parts = []
    if ctx:
        parts.append(f"{CTX} {ctx}")
    for role, text in hist:
        tag = USR if role == "user" else AST
        parts.append(f"{tag} {text}")
    return " ".join(parts).strip()


@torch.no_grad()
def generate_reply(history, ctx="", max_new_tokens=100, temperature=0.8, top_k=50):
    prompt = format_history(history, ctx=ctx)
    prompt_ids = tokenizer.encode(prompt).ids[-CONFIG["prompt_max_len"]:]
    gen_ids = model.generate(prompt_ids, bos_id=BOS_ID, eos_id=EOS_ID,
                              max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


# Example
demo_history = [("user", "Quelle est la capitale de la France ?")]
print("Assistant :", generate_reply(demo_history))

In [ ]:
# Interactive chat (write "exit" to stop)
history = []
print("Discussion avec le modèle (tape 'exit' pour quitter)\n")

while True:
    user_msg = input("Toi : ")
    if user_msg.strip().lower() in ("exit", "quit"):
        break
    history.append(("user", user_msg))
    reply = generate_reply(history)
    print("Assistant :", reply)
    history.append(("assistant", reply))